<a href="https://colab.research.google.com/github/hongqin/Python-CoLab-bootcamp/blob/master/tinyAI_emoji_guessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🧠 Tiny AI: Sentence → Emoji Mood Classifier
# Works in Google Colab. No extra installs.

import re
import random

# 1) A tiny "knowledge base" (words → points)
# Positive words add points; negative words subtract points.
lexicon = {
    # happy words
    "love": 2, "like": 1, "fun": 1, "happy": 2, "great": 2, "good": 1, "yay": 2,
    "awesome": 2, "cool": 1, "win": 1, "smile": 1, "sunny": 1, "kind": 1,
    # sad/grumpy words
    "sad": -2, "angry": -2, "mad": -2, "hate": -2, "tired": -1, "bored": -1,
    "bad": -1, "cry": -2, "lose": -1, "rainy": -1, "mean": -1, "uh": -1
}

EMOJI = { "happy": "🙂", "neutral": "😐", "sad": "🙁" }

# 2) Helper: clean text and split into simple lowercase words
def tokenize(text):
    text = text.lower()
    # keep letters, numbers, and spaces; turn everything else into space
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    # split on spaces
    return [w for w in text.split() if w]

# 3) Scoring: sum the points for words we know
def score_sentence(words):
    score = 0
    for w in words:
        score += lexicon.get(w, 0)
    return score

# 4) Decide label from score
def label_from_score(score, hi=2, lo=-2):
    if score >= hi:
        return "happy"
    elif score <= lo:
        return "sad"
    else:
        return "neutral"

# 5) Explain which words mattered (for teachable moments)
def highlight_hits(words):
    hits = []
    for w in words:
        if w in lexicon:
            pts = lexicon[w]
            sign = "+" if pts > 0 else ""
            hits.append(f"{w}({sign}{pts})")
    return ", ".join(hits) if hits else "no known mood words yet"

# 6) Tiny learning: ask user if the guess was correct; if not, learn a word
def learn_from_feedback(words, predicted):
    ans = input("Was the emoji right? (y/n) ").strip().lower()
    if ans != "n":
        return
    # Ask for the correct mood
    correct = ""
    while correct not in ("happy", "neutral", "sad"):
        correct = input("What should it be? Type happy / neutral / sad: ").strip().lower()
    # Let the student pick a word to teach
    # Suggest a word that isn't known yet, or any word from the sentence
    unknown = [w for w in words if w not in lexicon]
    if unknown:
        suggestion = random.choice(unknown)
        print(f"Let's teach it a word from your sentence. Suggested: '{suggestion}'")
    else:
        suggestion = words[0] if words else "okay"
        print(f"All words are known. We'll adjust one anyway. Suggested: '{suggestion}'")
    word = input(f"Type a word to teach (Enter to accept '{suggestion}'): ").strip().lower()
    if not word:
        word = suggestion

    # Map mood to points
    if correct == "happy": pts = 2
    elif correct == "sad": pts = -2
    else: pts = 0  # neutral
    lexicon[word] = pts
    print(f"Learned: '{word}' → {pts} points. Thanks! 🧠")

# 7) Run multiple rounds
print("🧠 Tiny AI: Emoji Mood Classifier")
print("Type a sentence; I'll guess 🙂 / 😐 / 🙁 . Type 'quit' to stop.")
print()

while True:
    sentence = input("Enter a sentence: ").strip()
    if sentence.lower() in ("quit", "exit", "q"):
        print("\nGoodbye! Here’s what I’ve learned today:")
        # show a small sample of the lexicon, sorted
        items = sorted(lexicon.items(), key=lambda x: (-abs(x[1]), x[0]))
        for w, pts in items[:20]:
            sign = "+" if pts > 0 else ""
            print(f"  {w:12s} {sign}{pts}")
        break

    words = tokenize(sentence)
    s = score_sentence(words)
    mood = label_from_score(s)
    print(f"AI says: {EMOJI[mood]}  ({mood.upper()}, score={s})")
    print("Words I noticed:", highlight_hits(words))
    learn_from_feedback(words, mood)
    print("-" * 40)

🧠 Tiny AI: Emoji Mood Classifier
Type a sentence; I'll guess 🙂 / 😐 / 🙁 . Type 'quit' to stop.

Enter a sentence: I love dot
AI says: 🙂  (HAPPY, score=2)
Words I noticed: love(+2)
